# CommitJL on Kaggle
Installs Go, builds the ZK circuit CLI, runs tests + smoke, then lets you run configured/sharded grids and produce tables/figures.

In [ ]:
import os, subprocess, sys
REPO = '/kaggle/working/code'  # expects the `code/` directory uploaded as a Kaggle Dataset, or cloned here
assert os.path.exists(REPO), f'{REPO} not found: attach the repo as a Kaggle Dataset or git-clone it here first'
os.chdir(REPO)
print(subprocess.run(['python', '--version'], capture_output=True, text=True).stdout)

In [ ]:
# Install Go 1.22 (tarball) to /kaggle/working/go -- no root/apt needed.
GO_VERSION = '1.22.5'
GO_DIR = '/kaggle/working/go'
if not os.path.exists(GO_DIR):
    url = f'https://go.dev/dl/go{GO_VERSION}.linux-amd64.tar.gz'
    subprocess.run(['curl', '-sSL', '-o', '/kaggle/working/go.tar.gz', url], check=True)
    subprocess.run(['tar', '-C', '/kaggle/working', '-xzf', '/kaggle/working/go.tar.gz'], check=True)
os.environ['PATH'] = f"{GO_DIR}/bin:" + os.environ['PATH']
os.environ['GOPATH'] = '/kaggle/working/gopath'
print(subprocess.run(['go', 'version'], capture_output=True, text=True).stdout)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

In [ ]:
# Build the ZK circuit CLI.
subprocess.run(['go', 'build', '-o', 'zk/bin/commitjl-prove', './cmd/commitjl-prove'], cwd='zk', check=True)

In [ ]:
# Quick pytest pass (skip slow/optional markers if any), then the smoke grid.
subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'], check=True)

In [ ]:
os.environ.setdefault('COMMITJL_DATA', '/kaggle/working/commitjl_data')
subprocess.run(['make', 'smoke'], check=True)

## Configurable cell -- pick a config and (optionally) a shard for this session

In [ ]:
CONFIG = 'experiments/configs/core.yaml'   # or native.yaml / prereg.yaml-derived custom config
SHARD = None   # e.g. '1/4' to split across 4 Kaggle sessions
SCRIPT = 'experiments.rq2_proof_cost'      # swap for rq1_fidelity / rq3_overhead / rq3_grinding / rq4_certificates / rq5_scaling / ablation

cmd = [sys.executable, '-m', SCRIPT, '--config', CONFIG]
if SHARD:
    cmd += ['--shard', SHARD]
subprocess.run(cmd, check=True)

In [ ]:
# Run additional shards/configs by re-running the cell above with different CONFIG/SHARD/SCRIPT values.
# Each run is resumable: rows already written are skipped.

## Tables, figures, and packaging results

In [ ]:
subprocess.run(['make', 'tables'], check=True)
subprocess.run(['make', 'figures'], check=True)
subprocess.run([sys.executable, '-m', 'experiments.summarize'], check=True)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/commitjl_results', 'zip', 'results')
print('wrote /kaggle/working/commitjl_results.zip')